In [2]:
import cv2
import pytesseract
import numpy as np
import pandas as pd
import re
import os
from datetime import datetime
import tkinter as tk
from tkinter import filedialog, messagebox, ttk
import threading

class SalesReportExtractor:
    def __init__(self):
        self.day_mapping = {
            'monday': 1, 'tuesday': 2, 'wednesday': 3, 'thursday': 4,
            'friday': 5, 'saturday': 6, 'sunday': 7,
            'mon': 1, 'tue': 2, 'wed': 3, 'thurs': 4,
            'fri': 5, 'sat': 6, 'sun': 7,
        }
        
        self.reason_mapping = {
            'f': 'Facebook', 'fb': 'Facebook',
            'm': 'Marketing',
            'live': 'Livestream', 'l': 'Livestream',
            'r1': 'Repeat Customer',
            'r2': 'Referral',
            'w': 'Walk-in',
            'appt': 'Appointment'
        }
        
        self.extracted_data = []
        self.unclear_markings = set()

    def test_basic_ocr(self, image_path):
        """Test if basic OCR is working at all"""
        try:
            img = cv2.imread(image_path)
            if img is None:
                print(f"Cannot load image: {image_path}")
                return ""
                
            gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
            
            # Try the most basic OCR
            text = pytesseract.image_to_string(gray)
            print(f"Basic OCR result length: {len(text)}")
            print(f"First 500 characters:\n{text[:500]}")
            print(f"Total lines: {len(text.splitlines())}")
            
            return text
        except Exception as e:
            print(f"Error in basic OCR test: {e}")
            return ""

    def preprocess_image_adaptive(self, image_path):
        """Improved preprocessing with multiple approaches"""
        img = cv2.imread(image_path)
        if img is None:
            print(f"Cannot load image: {image_path}")
            return None
        
        # Convert to grayscale
        gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
        
        # Test multiple preprocessing approaches
        approaches = []
        
        # Approach 1: Original grayscale
        approaches.append(("Original", gray))
        
        # Approach 2: Denoising + OTSU
        denoised = cv2.fastNlMeansDenoising(gray)
        _, otsu = cv2.threshold(denoised, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
        approaches.append(("Denoised_OTSU", otsu))
        
        # Approach 3: Adaptive threshold (Gaussian)
        adaptive_gaussian = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_GAUSSIAN_C, 
                                                cv2.THRESH_BINARY, 11, 2)
        approaches.append(("Adaptive_Gaussian", adaptive_gaussian))
        
        # Approach 4: Adaptive threshold (Mean)
        adaptive_mean = cv2.adaptiveThreshold(gray, 255, cv2.ADAPTIVE_THRESH_MEAN_C, 
                                            cv2.THRESH_BINARY, 11, 2)
        approaches.append(("Adaptive_Mean", adaptive_mean))
        
        # Approach 5: Morphological operations
        kernel = np.ones((2,2), np.uint8)
        morph = cv2.morphologyEx(otsu, cv2.MORPH_CLOSE, kernel)
        approaches.append(("Morphological", morph))
        
        # Test each approach and return the one that gives most meaningful text
        best_img = gray
        max_score = 0
        
        print(f"Testing {len(approaches)} preprocessing approaches...")
        
        for name, processed in approaches:
            try:
                # Quick test with basic config
                test_text = pytesseract.image_to_string(processed, config=r'--oem 3 --psm 6')
                
                # Score based on text length and number of meaningful lines
                meaningful_lines = [line.strip() for line in test_text.split('\n') 
                                  if line.strip() and len(line.strip()) > 2]
                score = len(test_text.strip()) + len(meaningful_lines) * 20
                
                print(f"{name}: {len(test_text)} chars, {len(meaningful_lines)} lines, score: {score}")
                
                if score > max_score:
                    max_score = score
                    best_img = processed
                    print(f"  ^ Best approach so far")
                    
            except Exception as e:
                print(f"Error testing {name}: {e}")
                continue
        
        return best_img

    def extract_text_from_image(self, image_path):
        """Enhanced text extraction with multiple OCR configurations"""
        try:
            # Get the best preprocessed image
            processed_img = self.preprocess_image_adaptive(image_path)
            if processed_img is None:
                return ""
            
            # Try multiple OCR configurations
            configs = [
                r'--oem 3 --psm 6',  # Default - uniform block of text
                r'--oem 3 --psm 4',  # Single column of text
                r'--oem 3 --psm 3',  # Fully automatic page segmentation
                r'--oem 3 --psm 8',  # Single word
                r'--oem 3 --psm 7',  # Single text line
                r'--oem 3 --psm 11', # Sparse text
                r'--oem 3 --psm 12', # Sparse text with OSD
            ]
            
            best_text = ""
            max_score = 0
            
            print(f"Testing {len(configs)} OCR configurations...")
            
            for config in configs:
                try:
                    text = pytesseract.image_to_string(processed_img, config=config)
                    
                    # Score based on text quality
                    meaningful_lines = [line.strip() for line in text.split('\n') 
                                      if line.strip() and len(line.strip()) > 2]
                    # Look for expected content (names, numbers, dates)
                    has_names = len(re.findall(r'\b[A-Z][A-Za-z]{2,}\b', text))
                    has_numbers = len(re.findall(r'\b\d{3,}\b', text))
                    has_dates = len(re.findall(r'\d{1,2}[/-]\d{1,2}[/-]\d{4}', text))
                    
                    score = (len(text.strip()) + 
                           len(meaningful_lines) * 10 + 
                           has_names * 15 + 
                           has_numbers * 10 + 
                           has_dates * 25)
                    
                    psm = config.split('--psm ')[1].split()[0]
                    print(f"PSM {psm}: {len(text)} chars, {len(meaningful_lines)} lines, "
                          f"{has_names} names, {has_numbers} numbers, score: {score}")
                    
                    if score > max_score:
                        max_score = score
                        best_text = text
                        print(f"  ^ Best configuration so far")
                        
                except Exception as e:
                    print(f"Error with config {config}: {e}")
                    continue
            
            print(f"Final best text length: {len(best_text)}")
            return best_text
            
        except Exception as e:
            print(f"Error extracting text from {image_path}: {e}")
            return ""

    def parse_date_and_day(self, text):
        """Enhanced date and day parsing"""
        # Multiple date patterns to try
        date_patterns = [
            r'Date\s*:?\s*(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})',
            r'(\d{1,2})[/\-.](\d{1,2})[/\-.](\d{4})',
            r'(\d{1,2})\s+(\d{1,2})\s+(\d{4})',
        ]
        
        # Day patterns
        day_patterns = [
            r'\(([A-Za-z]{3,})\)',
            r'([A-Za-z]{3,})\s*\)',
            r'Day\s*:?\s*([A-Za-z]+)',
            r'\b(monday|tuesday|wednesday|thursday|friday|saturday|sunday)\b',
            r'\b(mon|tue|wed|thu|fri|sat|sun)\b',
        ]
        
        # Try to find date
        formatted_date = None
        for pattern in date_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                try:
                    day, month, year = match.groups()
                    date_obj = datetime.strptime(f"{day}/{month}/{year}", "%d/%m/%Y")
                    formatted_date = date_obj.strftime("%Y-%m-%d")
                    print(f"Found date: {formatted_date}")
                    break
                except ValueError:
                    continue
        
        # Try to find day
        day_text = None
        day_num = None
        for pattern in day_patterns:
            match = re.search(pattern, text, re.IGNORECASE)
            if match:
                day_candidate = match.group(1).lower().strip()
                print(f"Found day candidate: {day_candidate}")
                
                # Map to our day system
                for day_key, num in self.day_mapping.items():
                    if day_key in day_candidate or day_candidate in day_key:
                        day_num = num
                        day_text = day_candidate.title()
                        print(f"Mapped to: {day_text} ({day_num})")
                        break
                
                if day_text:
                    break
        
        return formatted_date, day_text, day_num

    def parse_sales_data(self, text):
        """Enhanced sales data parsing with multiple strategies"""
        lines = text.split('\n')
        sales_data = []
        
        print(f"Parsing {len(lines)} lines for sales data...")
        
        # Multiple patterns for different data layouts
        patterns = [
            # Pattern 1: S/N Name SalesOrder Amount ... ReasonCode
            r'^\s*(\d+)\s+([A-Za-z]{2,})\s+([1-9]\d{5,7})\s+(\d{1,3}(?:[,.]\d{3})*(?:[,.]\d{2})?)\s+.*?([A-Z]{1,4})\s*$',
            
            # Pattern 2: Name SalesOrder Amount
            r'^\s*([A-Za-z]{2,})\s+([1-9]\d{5,7})\s+(\d{1,3}(?:[,.]\d{3})*(?:[,.]\d{2})?)',
            
            # Pattern 3: More flexible - any name followed by numbers
            r'^\s*([A-Z][A-Za-z]{1,})\s+.*?([1-9]\d{5,7}).*?(\d{1,3}(?:[,.]\d{3})*(?:[,.]\d{2})?)',
            
            # Pattern 4: Very flexible - just name and amount
            r'^\s*([A-Z][A-Za-z]+)\s+.*?(\d{3,}(?:[,.]\d{2})?)\s*',
        ]
        
        for i, line in enumerate(lines):
            line = line.strip()
            if not line or len(line) < 5:
                continue
                
            # Skip obvious header lines
            skip_keywords = ['s/n', 'name', 'sales order', 'amount', 'showroom', 'date', 'total']
            if any(keyword in line.lower() for keyword in skip_keywords):
                continue
            
            # Skip lines that are mostly symbols or numbers only
            if len(re.sub(r'[^A-Za-z]', '', line)) < 2:
                continue
            
            print(f"Analyzing line {i}: {line}")
            
            # Try each pattern
            for pattern_idx, pattern in enumerate(patterns):
                match = re.search(pattern, line, re.IGNORECASE)
                if match:
                    try:
                        groups = match.groups()
                        print(f"  Pattern {pattern_idx + 1} matched: {groups}")
                        
                        # Extract based on pattern
                        if len(groups) == 5:  # Pattern 1: has S/N and reason code
                            s_n, name, sales_order, amount, reason_code = groups
                        elif len(groups) == 3:  # Pattern 2 or 3
                            if groups[1].isdigit() and len(groups[1]) > 5:  # Has sales order
                                name, sales_order, amount = groups
                            else:  # Pattern 4 - just name and amount
                                name, amount = groups[0], groups[-1]
                                sales_order = self.extract_sales_order_from_context(line, lines, i)
                            reason_code = self.extract_reason_code(line, lines, i)
                        elif len(groups) == 2:  # Pattern 4
                            name, amount = groups
                            sales_order = self.extract_sales_order_from_context(line, lines, i)
                            reason_code = self.extract_reason_code(line, lines, i)
                        else:
                            continue
                        
                        # Clean and validate the extracted data
                        name = name.upper().strip()
                        if 'amount' in locals():
                            amount = re.sub(r'[^\d.]', '', str(amount))
                        if 'sales_order' not in locals():
                            sales_order = "Unknown"
                        if 'reason_code' not in locals():
                            reason_code = "Not specified"
                        
                        # Validate that we have meaningful data
                        if (name and len(name) >= 2 and 
                            amount and amount.replace('.', '').isdigit() and
                            float(amount) > 0):
                            
                            record = {
                                'name': name,
                                'sales_order_no': str(sales_order),
                                'amount': amount,
                                'reason_code': reason_code
                            }
                            
                            sales_data.append(record)
                            print(f"  ✓ Added record: {record}")
                            break  # Found valid data, move to next line
                        else:
                            print(f"  ✗ Invalid data: name='{name}', amount='{amount}'")
                            
                    except Exception as e:
                        print(f"  Error processing match: {e}")
                        continue
                else:
                    if pattern_idx == 0:  # Only print for first pattern to avoid spam
                        print(f"  No pattern match found")
        
        print(f"Extracted {len(sales_data)} sales records")
        return sales_data

    def extract_reason_code(self, current_line, all_lines, line_index):
        """Extract reason code from current line or nearby context"""
        # Check current line and nearby lines for reason codes
        search_lines = []
        for offset in range(-2, 3):  # Check 2 lines before and after
            idx = line_index + offset
            if 0 <= idx < len(all_lines):
                search_lines.append(all_lines[idx])
        
        # Reason code patterns
        reason_patterns = [
            r'\b([FWM]|FB|LIVE|R1|R2|APPT)\b',
            r'\b(Facebook|Marketing|Livestream|Repeat|Referral|Walk[-\s]?in|Appointment)\b'
        ]
        
        for line in search_lines:
            for pattern in reason_patterns:
                matches = re.findall(pattern, line, re.IGNORECASE)
                if matches:
                    code = matches[0].upper().strip()
                    mapped = self.reason_mapping.get(code.lower(), code)
                    if mapped != code:
                        return mapped
                    else:
                        # Add to unclear markings for manual review
                        self.unclear_markings.add(code)
                        return f"UNCLEAR: {code}"
        
        return "Not specified"

    def extract_sales_order_from_context(self, current_line, all_lines, line_index):
        """Extract sales order number from context"""
        # Look for 6-7 digit numbers in current and nearby lines
        for offset in range(-1, 2):
            idx = line_index + offset
            if 0 <= idx < len(all_lines):
                numbers = re.findall(r'\b[1-9]\d{5,7}\b', all_lines[idx])
                if numbers:
                    return numbers[0]
        
        return "Unknown"

    def process_image(self, image_path):
        """Main image processing method with comprehensive error handling"""
        print(f"\n=== PROCESSING: {os.path.basename(image_path)} ===")
        
        try:
            # Step 1: Basic validation
            if not os.path.exists(image_path):
                print(f"File does not exist: {image_path}")
                return []
            
            # Step 2: Test basic OCR capability
            basic_text = self.test_basic_ocr(image_path)
            if not basic_text.strip():
                print("Basic OCR failed - no text extracted")
                return []
            
            # Step 3: Enhanced text extraction
            print("Running enhanced text extraction...")
            text = self.extract_text_from_image(image_path)
            
            if not text.strip():
                print("Enhanced OCR also failed")
                return []
            
            print(f"Extracted text preview:\n{text[:300]}...")
            
            # Step 4: Parse date and day
            date, day_text, day_num = self.parse_date_and_day(text)
            
            # Step 5: Parse sales data
            sales_data = self.parse_sales_data(text)
            
            # Step 6: Add metadata to each record
            for record in sales_data:
                record['date'] = date or "Unknown"
                record['day'] = day_text or "Unknown"
                record['day_number'] = day_num or 0
                record['source_file'] = os.path.basename(image_path)
            
            print(f"Successfully processed: {len(sales_data)} records extracted")
            return sales_data
            
        except Exception as e:
            print(f"Error processing {image_path}: {str(e)}")
            import traceback
            traceback.print_exc()
            return []

    def process_multiple_images(self, image_paths):
        """Process multiple images and combine results"""
        all_data = []
        
        for image_path in image_paths:
            data = self.process_image(image_path)
            all_data.extend(data)
        
        self.extracted_data = all_data
        return all_data

    def export_to_csv(self, output_path):
        """Export extracted data to CSV with proper column ordering"""
        if not self.extracted_data:
            print("No data to export")
            return False
        
        try:
            df = pd.DataFrame(self.extracted_data)
            
            # Reorder columns for better readability
            column_order = ['date', 'day', 'day_number', 'name', 'sales_order_no', 
                          'amount', 'reason_code', 'source_file']
            
            # Only include columns that exist in the data
            available_columns = [col for col in column_order if col in df.columns]
            df = df[available_columns]
            
            # Export to CSV
            df.to_csv(output_path, index=False)
            print(f"Data exported to: {output_path}")
            
            # Print summary of unclear markings
            if self.unclear_markings:
                print("\nUnclear markings found (please clarify):")
                for marking in sorted(self.unclear_markings):
                    print(f"  - {marking}")
            
            return True
            
        except Exception as e:
            print(f"Error exporting data: {e}")
            return False


class SalesExtractorGUI:
    def __init__(self, root):
        self.root = root
        self.root.title("💖 Sales Report OCR Extractor 💖")
        self.root.geometry("1000x800")
        
        self.extractor = SalesReportExtractor()
        self.selected_files = []
        
        # Create theme
        self.colors = self.create_theme()
        self.root.configure(bg=self.colors['light_pink'])
        
        # Create widgets
        self.create_widgets()

    def create_theme(self):
        style = ttk.Style()
        
        # Define pink color palette
        colors = {
            'light_pink': '#FFE1E6',      # Very light pink background
            'soft_pink': '#F8BBD9',       # Soft pink for buttons
            'medium_pink': '#E4007C',     # Medium pink for accents
            'dark_pink': '#C21460',       # Dark pink for text
            'white': '#FFFFFF',           # Pure white
            'light_gray': '#F5F5F5'       # Light gray for contrast
        }
        
        # Configure main frame style
        style.configure('TFrame', background=colors['light_pink'])
        
        # Configure button style with hearts
        style.configure('Pink.TButton',
                       background=colors['soft_pink'],
                       foreground=colors['dark_pink'],
                       borderwidth=2,
                       focuscolor='none',
                       font=('Arial', 10, 'bold'))
        
        style.map('Pink.TButton',
                 background=[('active', colors['medium_pink']),
                            ('pressed', colors['dark_pink'])],
                 foreground=[('active', colors['white']),
                            ('pressed', colors['white'])])
        
        # Configure labelframe styles
        style.configure('TLabelFrame',
                       background=colors['light_pink'],
                       foreground=colors['dark_pink'],
                       borderwidth=2,
                       relief='groove')
        
        style.configure('TLabelFrame.Label',
                       background=colors['light_pink'],
                       foreground=colors['dark_pink'],
                       font=('Arial', 11, 'bold'))
        
        # Configure label styles
        style.configure('Title.TLabel',
                       background=colors['light_pink'],
                       foreground=colors['dark_pink'],
                       font=('Arial', 18, 'bold'))
        
        style.configure('TLabel',
                       background=colors['light_pink'],
                       foreground=colors['dark_pink'],
                       font=('Arial', 10))
        
        # Configure progressbar
        style.configure('Pink.Horizontal.TProgressbar',
                       background=colors['medium_pink'],
                       troughcolor=colors['light_pink'],
                       borderwidth=2,
                       lightcolor=colors['soft_pink'],
                       darkcolor=colors['dark_pink'])
        
        return colors

    def create_widgets(self):
        # Main frame
        main_frame = ttk.Frame(self.root, padding="15")
        main_frame.grid(row=0, column=0, sticky=(tk.W, tk.E, tk.N, tk.S))
        
        # Title
        title_label = ttk.Label(main_frame, 
                               text="💖 Sales Report OCR Data Extractor 💖", 
                               style='Title.TLabel')
        title_label.grid(row=0, column=0, columnspan=2, pady=(0, 30))
        
        # File selection frame
        file_frame = tk.LabelFrame(main_frame,
                                  text="📁 Select Your Images",
                                  padding="15")
        file_frame.pack(fill=tk.X, pady=(0, 20))
        
        # Select files button
        select_btn = tk.Button(file_frame,
                              text="Choose Images",
                              command=self.select_files,
                              bg=self.colors['soft_pink'],
                              fg=self.colors['dark_pink'],
                              font=('Arial', 11, 'bold'),
                              padx=20, pady=8,
                              relief='raised',
                              borderwidth=2)
        select_btn.pack(side=tk.LEFT, padx=(0, 15))
        
        # File count label
        self.file_count_label = tk.Label(file_frame,
                                        text="No files selected yet",
                                        bg=self.colors['light_pink'],
                                        fg=self.colors['medium_pink'],
                                        font=('Arial', 10))
        self.file_count_label.pack(side=tk.LEFT)
        
        # File list
        list_frame = tk.Frame(file_frame, bg=self.colors['light_pink'])
        list_frame.pack(fill=tk.BOTH, expand=True, pady=(15, 0))
        
        self.file_listbox = tk.Listbox(list_frame,
                                      height=6,
                                      bg=self.colors['white'],
                                      fg=self.colors['dark_pink'],
                                      font=('Arial', 10),
                                      selectbackground=self.colors['soft_pink'],
                                      selectforeground=self.colors['dark_pink'],
                                      borderwidth=2,
                                      relief='groove')
        self.file_listbox.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        # Scrollbar for listbox
        list_scrollbar = tk.Scrollbar(list_frame, orient="vertical", 
                                     command=self.file_listbox.yview)
        self.file_listbox.configure(yscrollcommand=list_scrollbar.set)
        list_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Process button
        self.process_btn = tk.Button(main_frame,
                                    text="Process Images",
                                    command=self.process_images_threaded,
                                    state="disabled",
                                    bg=self.colors['soft_pink'],
                                    fg=self.colors['dark_pink'],
                                    font=('Arial', 12, 'bold'),
                                    padx=30, pady=10,
                                    relief='raised',
                                    borderwidth=3)
        self.process_btn.pack(pady=(0, 20))
        
        # Progress bar
        self.progress = ttk.Progressbar(main_frame,
                                       mode='indeterminate',
                                       length=400)
        self.progress.pack(pady=(0, 20))
        
        # Results frame
        results_frame = tk.LabelFrame(main_frame,
                                     text="Processing Results",
                                     font=('Arial', 12, 'bold'),
                                     bg=self.colors['light_pink'],
                                     fg=self.colors['dark_pink'],
                                     padx=15, pady=15)
        results_frame.pack(fill=tk.BOTH, expand=True, pady=(0, 20))
        
        # Results text area
        text_frame = tk.Frame(results_frame, bg=self.colors['light_pink'])
        text_frame.pack(fill=tk.BOTH, expand=True)
        
        self.results_text = tk.Text(text_frame,
                                   height=15,
                                   width=80,
                                   bg=self.colors['white'],
                                   fg=self.colors['dark_pink'],
                                   font=('Arial', 10),
                                   borderwidth=2,
                                   relief='groove',
                                   wrap=tk.WORD)
        self.results_text.pack(side=tk.LEFT, fill=tk.BOTH, expand=True)
        
        # Scrollbar for results
        results_scrollbar = tk.Scrollbar(text_frame, orient="vertical",
                                        command=self.results_text.yview)
        self.results_text.configure(yscrollcommand=results_scrollbar.set)
        results_scrollbar.pack(side=tk.RIGHT, fill=tk.Y)
        
        # Export button
        self.export_btn = tk.Button(main_frame,
                                   text="Export to CSV",
                                   command=self.export_data,
                                   state="disabled",
                                   bg=self.colors['success'],
                                   fg='white',
                                   font=('Arial', 11, 'bold'),
                                   padx=25, pady=8,
                                   relief='raised',
                                   borderwidth=2)
        self.export_btn.pack(pady=(0, 10))
        
        # Footer
        footer_label = tk.Label(main_frame,
                               text="For sales data analysis",
                               font=('Arial', 9, 'italic'),
                               bg=self.colors['light_pink'],
                               fg=self.colors['medium_pink'])
        footer_label.pack()

    def update_file_count(self, count):
        if count == 0:
            message = "No files selected yet"
        elif count == 1:
            message = "1 file selected"
        else:
            message = f"{count} files selected"
        
        self.file_count_label.config(text=message)

    def show_message(self, message, color=None):
        if color is None:
            color = self.colors['dark_pink']
        self.results_text.insert(tk.END, f"{message}\n")
        self.results_text.see(tk.END)
        self.root.update()

    def select_files(self):
        file_types = [
            ("Image files", "*.png *.jpg *.jpeg *.bmp *.tiff *.tif"),
            ("All files", "*.*")
        ]
        
        files = filedialog.askopenfilenames(
            title="Select your sales report images",
            filetypes=file_types
        )
        
        if files:
            self.selected_files = list(files)
            self.update_file_count(len(files))
            
            # Update listbox
            self.file_listbox.delete(0, tk.END)
            for i, file in enumerate(files):
                filename = os.path.basename(file)
                self.file_listbox.insert(tk.END, f"{i+1}. {filename}")
            
            self.process_btn.config(state="normal")
            self.show_message(f"Selected {len(files)} image{'s' if len(files) != 1 else ''}")

    def process_images_threaded(self):
        """Run processing in a separate thread to avoid freezing the UI"""
        thread = threading.Thread(target=self.process_images)
        thread.daemon = True
        thread.start()

    def process_images(self):
        if not self.selected_files:
            messagebox.showwarning("No Files", "Please select some images first!")
            return

        # Start processing
        self.progress.start()
        self.process_btn.config(state="disabled")
        self.results_text.delete(1.0, tk.END)

        self.show_message("Starting OCR processing...")
        
        try:
            all_data = []
            successful_files = 0
            
            for i, image_path in enumerate(self.selected_files):
                filename = os.path.basename(image_path)
                self.show_message(f"\nProcessing file {i+1}/{len(self.selected_files)}: {filename}")
                
                data = self.extractor.process_image(image_path)
                if data:
                    all_data.extend(data)
                    successful_files += 1
                    self.show_message(f"✓ Extracted {len(data)} records from {filename}", 
                                    self.colors['success'])
                else:
                    self.show_message(f"✗ No data extracted from {filename}", 
                                    self.colors['error'])

            # Final results
            self.show_message(f"\n=== PROCESSING COMPLETE ===")
            self.show_message(f"Successfully processed: {successful_files}/{len(self.selected_files)} files")
            self.show_message(f"Total records extracted: {len(all_data)}")
            
            if all_data:
                # Store the data
                self.extractor.extracted_data = all_data
                
                # Display sample results
                self.show_message(f"\nSample data preview:")
                for i, record in enumerate(all_data[:3]):  # Show first 3 records
                    self.show_message(f"{i+1}. {record}")
                
                if len(all_data) > 3:
                    self.show_message(f"... and {len(all_data)-3} more records")
                
                self.export_btn.config(state="normal")
                self.show_message(f"\nData is ready for export!")
                
                # Show unclear markings if any
                if self.extractor.unclear_markings:
                    self.show_message(f"\nUnclear markings found (needs manual review):")
                    for marking in sorted(self.extractor.unclear_markings):
                        self.show_message(f"  - {marking}")
            else:
                self.show_message(f"\nNo data could be extracted from any images.")
                self.show_message(f"Troubleshooting suggestions:")
                self.show_message(f"• Ensure images are clear and high resolution")
                self.show_message(f"• Check that images contain readable text")
                self.show_message(f"• Try scanning images at higher DPI if possible")
                self.show_message(f"• Ensure good lighting and contrast in original documents")

        except Exception as e:
            self.show_message(f"Processing error: {str(e)}", self.colors['error'])
            messagebox.showerror("Error", f"Something went wrong during processing:\n{str(e)}")

        finally:
            # Stop progress bar and re-enable button
            self.progress.stop()
            self.process_btn.config(state="normal")

    def export_data(self):
        if not self.extractor.extracted_data:
            messagebox.showwarning("No Data", "No data to export yet!")
            return
        
        output_file = filedialog.asksaveasfilename(
            title="Save your data",
            defaultextension=".csv",
            filetypes=[("CSV files", "*.csv"), ("All files", "*.*")]
        )
        
        if output_file:
            success = self.extractor.export_to_csv(output_file)
            if success:
                messagebox.showinfo("Success!", 
                    f"Your data has been exported successfully!\n\nSaved to:\n{output_file}")
                self.show_message(f"Data exported to: {os.path.basename(output_file)}")
            else:
                messagebox.showerror("Export Failed", "Something went wrong while saving your data!")


def check_dependencies():
    """Check if required packages are installed"""
    try:
        import pytesseract
        import cv2
        import pandas as pd
        return True
    except ImportError as e:
        print(f"Required package not found: {e}")
        print("Please install required packages:")
        print("pip install pytesseract opencv-python pandas pillow")
        return False


def main():
    # Check if required packages are installed
    if not check_dependencies():
        return
    
    # Check if Tesseract is available
    try:
        import pytesseract
        # Try to get Tesseract version to verify it's installed
        pytesseract.get_tesseract_version()
    except Exception as e:
        print(f"Tesseract OCR not found: {e}")
        print("Please install Tesseract OCR:")
        print("Windows: Download from https://github.com/UB-Mannheim/tesseract/wiki")
        print("Mac: brew install tesseract")
        print("Linux: sudo apt-get install tesseract-ocr")
        return
    
    # Create and run the GUI
    root = tk.Tk()
    app = SalesExtractorGUI(root)
    root.mainloop()


if __name__ == "__main__":
    main()

TclError: unknown option "-padding"